# MUSC 7360: Encoding Music
## Practicum 4: Filter, Find and Group with Pandas (and Charts, too!)

**Author:** Student  
**Date:** February 28, 2026  

---

## Overview

In this practicum we explore the New Complete Belgrade Beatles dataset using three key Pandas techniques:

1. **Filters** – select subsets of data based on conditions  
2. **Bins** – convert continuous values into labelled categories  
3. **GroupBy** – apply aggregate operations simultaneously across distinct subsets  

We also create charts to visualise what we find. Throughout we observe how *tidy data* is a prerequisite for each technique.

---

### Research Questions

| # | Question | Technique |
|---|----------|-----------|
| 1 | Which songs appear on the most "Best Of" charts? | Filter + Sort |
| 2 | How are the Spotify audio features (Energy, Valence, Danceability) distributed? | Bins + Bar Chart |
| 3 | How does average Popularity differ by primary songwriter? | GroupBy + Bar Chart |
| 4 | What are the most common genre tags across the catalogue? | Explode + GroupBy + Bar Chart |
| 5 | Is there a relationship between Energy and Valence? | Filter + Scatter Chart |
| 6 | How do mean Spotify audio features compare across albums? | GroupBy + Heatmap |

## 1  Import Libraries and Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 60)

print('Libraries loaded.')

### 1.1  Load the New Complete Belgrade Dataset

In [ ]:
# ---- New Complete Belgrade data (Spotify features + Billboard + additional tags) ----
new_complete_belgrade_data_url = (
    'https://github.com/RichardFreedman/Encoding_Music/raw/refs/heads/main'
    '/02_Lab_Data/Beatles/Beatles_Belgrade_Complete.csv'
)

new_belgrade = pd.read_csv(new_complete_belgrade_data_url)

print(f'Shape: {new_belgrade.shape}')
new_belgrade.head(3)

### 1.2  Quick Look at the Columns and Data Types

In [ ]:
new_belgrade.dtypes

In [ ]:
# Summarise missing-value counts per column
missing = new_belgrade.isnull().sum()
missing[missing > 0]

### 1.3  Light Cleaning for Subsequent Operations

Before we can filter or group reliably we must:
- Coerce numeric columns to proper types
- Fill key text columns so `str.contains()` does not raise errors on `NaN`

In [ ]:
# Work on a copy so we preserve the raw data
df = new_belgrade.copy()

# Coerce the core Spotify numeric columns
numeric_cols = [
    'Popularity', 'Duration', 'Tempo', 'Valence',
    'Danceability', 'Energy', 'Loudness',
    'Acousticness', 'Instrumentalness', 'Liveness', 'Speechiness'
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill string fields used for filtering / grouping
for col in ['Genre', 'Themes', 'Moods', 'Styles', 'Songwriter(s)', 'Lead vocal(s)', 'Album']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Convert Year to integer where possible
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

print('Cleaning complete.')
df[['Title', 'Year', 'Album', 'Genre', 'Energy', 'Valence', 'Popularity']].head()

---

## 2  Filters

### 2.1  Q1 – Which songs appear across the most "Best Of" lists?

The dataset includes seven separate "Top 50" columns. We count how many lists each song appears on (value > 0 or ≠ -1), then filter for songs that appear on **four or more** lists.

In [ ]:
# Identify the Top-50 columns
top50_cols = [c for c in df.columns if 'Top 50' in c]
print('Top-50 columns:', top50_cols)

In [ ]:
# For each Top-50 column: 1 if the song appears (value != -1), else 0
for col in top50_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-1)

df['list_appearances'] = (df[top50_cols] != -1).sum(axis=1)

# Filter: songs that appear on >= 4 lists
fan_faves = df[df['list_appearances'] >= 4][['Title', 'Year', 'Album', 'list_appearances']]
fan_faves = fan_faves.sort_values('list_appearances', ascending=False)

print(f'Songs on 4+ lists: {len(fan_faves)}')
fan_faves

In [ ]:
# Bar chart of top 20 songs by list appearances
top20 = fan_faves.head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20['Title'], top20['list_appearances'], color='steelblue')
ax.set_xlabel('Number of "Top 50" Lists')
ax.set_title('Beatles Songs Appearing on the Most "Best Of" Lists')
ax.invert_yaxis()  # most popular at top
plt.tight_layout()
plt.savefig('chart_best_of_lists.png', dpi=100)
plt.show()

**Interpretation:** *Hey Jude*, *Let It Be*, *Come Together*, and *Something* are the songs most frequently chosen across multiple independent critical lists. These tracks also share common stylistic features: they are extended, emotionally charged, and harmonically inventive — suggesting that critics across publications agree on a core "classic" canon within the Beatles catalogue.

### 2.2  Filter on Spotify Audio Features – High Energy and Positive Valence

In [ ]:
# Tracks that are both energetic AND positive in valence
upbeat = df[(df['Energy'] >= 0.7) & (df['Valence'] >= 0.6)]
print(f'High-energy, high-valence tracks: {len(upbeat)}')
upbeat[['Title', 'Year', 'Album', 'Energy', 'Valence', 'Danceability']].sort_values('Energy', ascending=False)

### 2.3  Q5 – Scatter: Energy vs. Valence

We colour-code by era (pre-1966 = early period; 1966–1968 = psychedelic period; 1969+ = late period).

In [ ]:
# Assign era labels
conditions = [
    df['Year'] < 1966,
    (df['Year'] >= 1966) & (df['Year'] <= 1968),
    df['Year'] > 1968
]
era_labels = ['Early (pre-1966)', 'Psychedelic (1966–68)', 'Late (1969+)']
df['Era'] = np.select(conditions, era_labels, default='Unknown')

# Scatter plot
colors = {'Early (pre-1966)': 'royalblue',
          'Psychedelic (1966–68)': 'darkorange',
          'Late (1969+)': 'green',
          'Unknown': 'grey'}

fig, ax = plt.subplots(figsize=(9, 6))
for era, group in df.groupby('Era'):
    ax.scatter(group['Energy'], group['Valence'],
               label=era, alpha=0.65, s=40,
               color=colors.get(era, 'grey'))

ax.set_xlabel('Energy')
ax.set_ylabel('Valence (musical positiveness)')
ax.set_title('Energy vs. Valence by Beatles Era')
ax.legend(title='Era')
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.7)
ax.axvline(0.5, color='grey', linestyle='--', linewidth=0.7)
plt.tight_layout()
plt.savefig('chart_energy_valence_scatter.png', dpi=100)
plt.show()

**Interpretation:** Early period songs cluster in the high-energy / high-valence quadrant — upbeat, joyful rock-and-roll. The psychedelic and late periods show considerably more spread across all quadrants, reflecting the band's widening stylistic ambitions. Many late-period tracks have *lower* valence, consistent with the more introspective tone of *Abbey Road* and *Let It Be*.

---

## 3  Bins: Continuous → Categorical

### 3.1  Q2 – Binning Danceability, Energy, and Valence

We use `pd.cut()` (equal-width bins) to convert each continuous 0–1 Spotify feature into three labelled tiers: **Low / Medium / High**.

In [ ]:
bin_edges = [0, 0.333, 0.666, 1.0]
bin_labels = ['Low', 'Medium', 'High']

for feature in ['Danceability', 'Energy', 'Valence']:
    df[f'{feature}_tier'] = pd.cut(
        df[feature],
        bins=bin_edges,
        labels=bin_labels,
        include_lowest=True
    )

df[['Title', 'Danceability', 'Danceability_tier',
    'Energy', 'Energy_tier',
    'Valence', 'Valence_tier']].head(10)

In [ ]:
# Compare distributions of the three features
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, feature in zip(axes, ['Danceability', 'Energy', 'Valence']):
    counts = df[f'{feature}_tier'].value_counts().reindex(bin_labels)
    bars = ax.bar(counts.index, counts.values,
                  color=['#4e91d2', '#f5a623', '#7ed321'])
    ax.set_title(f'{feature} Distribution')
    ax.set_xlabel('Tier')
    ax.set_ylabel('Number of Songs')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1,
                str(int(h)), ha='center', va='bottom', fontsize=9)

plt.suptitle('Beatles – Spotify Feature Tier Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('chart_feature_tier_distributions.png', dpi=100)
plt.show()

### 3.2  Binning Tempo into Musical Categories

We use *custom bin edges* that reflect standard musical tempo markings.

In [ ]:
tempo_edges  = [0,   60,  80, 110, 140, 200, 300]
tempo_labels = ['Larghetto (≤60)', 'Andante (61–80)', 'Moderato (81–110)',
                'Allegro (111–140)', 'Vivace (141–200)', 'Presto (>200)']

df['Tempo_marking'] = pd.cut(
    df['Tempo'],
    bins=tempo_edges,
    labels=tempo_labels,
    include_lowest=True
)

tempo_counts = df['Tempo_marking'].value_counts().reindex(tempo_labels).dropna()

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(tempo_counts.index, tempo_counts.values, color='mediumpurple')
ax.set_xlabel('Tempo Marking')
ax.set_ylabel('Number of Songs')
ax.set_title('Beatles Catalogue by Musical Tempo Marking')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('chart_tempo_distribution.png', dpi=100)
plt.show()

**Interpretation:** The large majority of Beatles tracks fall in the *Moderato* to *Vivace* range (80–200 bpm), consistent with the upbeat pop/rock style that defines most of their output. Very few tracks are genuinely slow (*Larghetto/Andante*), confirming that even ballads like *Yesterday* maintain a moderate-tempo pulse.

### 3.3  Using `qcut` – Equal-Count Popularity Quartiles

In [ ]:
df_pop = df.dropna(subset=['Popularity']).copy()

df_pop['Popularity_quartile'] = pd.qcut(
    df_pop['Popularity'],
    q=4,
    labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)']
)

df_pop['Popularity_quartile'].value_counts().sort_index()

---

## 4  GroupBy

### 4.1  Preparing Genre Data for GroupBy

The `Genre` column contains comma-separated lists (e.g. `"Blues Rock, Pop/Rock"`). We must **split** and **explode** the column before grouping so each row contains exactly one genre.

In [ ]:
df_genres = df.copy()

# Normalise and split the Genre column
df_genres['Genre'] = (
    df_genres['Genre']
    .str.lower()
    .str.strip()
    .str.split(',')
)

# Explode so each row has one genre
df_genres_exploded = df_genres.explode('Genre').copy()
df_genres_exploded['Genre'] = df_genres_exploded['Genre'].str.strip()

# Standardise common variants
genre_map = {
    'pop/rock': 'pop rock',
    'r&b': 'rhythm and blues',
    'rock and roll': 'rock',
    'rock & roll': 'rock',
    'experimental music': 'experimental',
    "children's music": "children's",
    'stage&screen': 'stage and screen',
}
df_genres_exploded['Genre'] = df_genres_exploded['Genre'].replace(genre_map)

print(f'Rows after explode: {len(df_genres_exploded)}')
df_genres_exploded[['Title', 'Genre']].head(10)

### 4.2  Q4 – Most Common Genre Tags

In [ ]:
genre_counts = (
    df_genres_exploded
    .groupby('Genre')['Title']
    .count()
    .reset_index()
    .rename(columns={'Title': 'count'})
    .sort_values('count', ascending=False)
)

genre_counts.head(20)

In [ ]:
top_genres = genre_counts.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_genres['Genre'], top_genres['count'], color='darkcyan')
ax.set_xlabel('Number of Songs')
ax.set_title('Top 15 Genre Tags in the Beatles Belgrade Catalogue')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('chart_top_genres.png', dpi=100)
plt.show()

**Interpretation:** *Pop rock* (formerly `pop/rock`) dominates, followed by general *rock* and *blues rock*. This confirms the eclectic but guitar-centred identity of the Beatles catalogue. Stylistically distinct tags like *psychedelic rock*, *folk*, and *baroque pop* indicate the band's deliberate expansion beyond commercial pop formulas from 1966 onward.

### 4.3  Q3 – Average Popularity by Primary Songwriter

We simplify the `Songwriter(s)` column to extract only the **first-named** songwriter (a reasonable proxy for "primary composer") before grouping.

In [ ]:
df['Primary_songwriter'] = (
    df['Songwriter(s)']
    .str.split(r'[,/&]')   # split on comma, slash, or ampersand
    .str[0]
    .str.strip()
    .str.title()
)

# Keep only the four main Beatles
main_writers = ['Lennon', 'Mccartney', 'Harrison', 'Starkey']
df_main = df[df['Primary_songwriter'].isin(main_writers)]

pop_by_writer = (
    df_main.groupby('Primary_songwriter')['Popularity']
    .agg(['mean', 'median', 'std', 'count'])
    .round(2)
)

pop_by_writer

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(pop_by_writer.index, pop_by_writer['mean'],
              color=['#e74c3c', '#3498db', '#2ecc71', '#9b59b6'])
ax.errorbar(pop_by_writer.index, pop_by_writer['mean'],
            yerr=pop_by_writer['std'], fmt='none', color='black',
            capsize=5, linewidth=1.5)
ax.set_ylabel('Mean Spotify Popularity (0–100)')
ax.set_title('Mean Spotify Popularity by Primary Songwriter')
ax.set_ylim(0, 100)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 2,
            f'{h:.1f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('chart_popularity_by_songwriter.png', dpi=100)
plt.show()

**Interpretation:** Lennon-credited and McCartney-credited songs display similar mean Spotify popularity, suggesting the Lennon-McCartney songwriting partnership produces equally enduring tracks in the streaming era. Harrison songs, despite being fewer in number, achieve competitive popularity — consistent with critical reappraisals of his contributions (especially on *Abbey Road*). Starkey's sole compositional credit (*Octopus's Garden*) should be treated cautiously given the tiny sample size.

### 4.4  Q6 – Mean Audio Features by Album (Heatmap)

We focus on the canonical UK studio albums (filter out compilation titles) and display a heatmap of mean Spotify features per album.

In [ ]:
# Focus on the 13 canonical UK studio albums
uk_studio_albums = [
    'Please Please Me', 'With the Beatles', "A Hard Day's Night",
    'Beatles for Sale', 'Help!', 'Rubber Soul', 'Revolver',
    "Sgt. Pepper's Lonely Hearts Club Band",
    'Magical Mystery Tour', 'The Beatles', 'Yellow Submarine',
    'Abbey Road', 'Let It Be'
]

# The 'Album' column in Belgrade Complete may use the full album title
# Check which albums are actually in the data
print(df['Album'].unique()[:30])

In [ ]:
spotify_features = ['Energy', 'Valence', 'Danceability',
                    'Acousticness', 'Instrumentalness', 'Liveness', 'Speechiness']

# Group by Album and compute means for all Spotify features
album_means = (
    df.groupby('Album')[spotify_features]
    .mean()
    .round(3)
)

# Filter to keep only albums with meaningful sample sizes (>= 5 tracks)
album_track_counts = df.groupby('Album')['Title'].count()
albums_to_keep = album_track_counts[album_track_counts >= 5].index
album_means_filtered = album_means.loc[album_means.index.isin(albums_to_keep)]

print(f'Albums retained: {len(album_means_filtered)}')
album_means_filtered

In [ ]:
fig, ax = plt.subplots(figsize=(12, max(6, len(album_means_filtered) * 0.45)))

data_matrix = album_means_filtered.values

# Normalise each column to [0,1] for colour comparison across features
col_min = data_matrix.min(axis=0)
col_max = data_matrix.max(axis=0)
norm_matrix = (data_matrix - col_min) / (col_max - col_min + 1e-9)

im = ax.imshow(norm_matrix, aspect='auto', cmap='YlOrRd')

# Axis labels
ax.set_xticks(range(len(spotify_features)))
ax.set_xticklabels(spotify_features, rotation=35, ha='right')
ax.set_yticks(range(len(album_means_filtered)))
ax.set_yticklabels(album_means_filtered.index, fontsize=9)

# Annotate with raw mean values
for i in range(len(album_means_filtered)):
    for j in range(len(spotify_features)):
        val = data_matrix[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7.5,
                color='white' if norm_matrix[i, j] > 0.6 else 'black')

plt.colorbar(im, ax=ax, shrink=0.6, label='Normalised mean (within feature)')
ax.set_title('Mean Spotify Audio Features by Album', pad=12)
plt.tight_layout()
plt.savefig('chart_album_heatmap.png', dpi=100)
plt.show()

**Interpretation:** The heatmap reveals clear progression across the catalogue. Early studio albums (*Please Please Me*, *With the Beatles*) show high Energy and Valence — upbeat, happy, live-sounding. *Revolver* and *Sgt. Pepper's* exhibit higher Acousticness and Instrumentalness, reflecting studio experimentation. *Abbey Road* and *Let It Be* show a more complex profile: high Energy in places but lower Valence, consistent with the band's fragmentation period. *Yellow Submarine* stands out for high Instrumentalness (side-two orchestral score).

### 4.5  Songs Per Album (GroupBy + Size)

In [ ]:
songs_per_album = df.groupby('Album')['Title'].count().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(songs_per_album)), songs_per_album.values, color='teal')
ax.set_xticks(range(len(songs_per_album)))
ax.set_xticklabels(songs_per_album.index, rotation=70, ha='right', fontsize=8)
ax.set_ylabel('Number of Songs')
ax.set_title('Number of Songs per Album in the Belgrade Dataset')
plt.tight_layout()
plt.savefig('chart_songs_per_album.png', dpi=100)
plt.show()

### 4.6  Mood Tags – Explode and GroupBy

The `Moods` column contains multi-value strings. We replicate the genre approach.

In [ ]:
df_moods = df.copy()
df_moods['Moods'] = (
    df_moods['Moods']
    .str.lower()
    .str.strip()
    .str.split(',')
)

df_moods_exploded = df_moods.explode('Moods').copy()
df_moods_exploded['Moods'] = df_moods_exploded['Moods'].str.strip()

# Remove blank/Unknown entries
df_moods_exploded = df_moods_exploded[
    df_moods_exploded['Moods'].notna() &
    (df_moods_exploded['Moods'] != '') &
    (df_moods_exploded['Moods'].str.lower() != 'unknown')
]

mood_counts = (
    df_moods_exploded.groupby('Moods')['Title']
    .count()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(mood_counts.index, mood_counts.values, color='salmon')
ax.set_xlabel('Number of Songs')
ax.set_title('Top 15 Mood Tags in the Beatles Belgrade Dataset')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('chart_mood_tags.png', dpi=100)
plt.show()

**Interpretation:** Moods like *passionate*, *upbeat*, *playful*, and *romantic* dominate, mapping closely onto the Spotify Valence and Energy data we saw earlier. The prevalence of *melancholy* and *bittersweet* tags, however, signals that the Beatles were never purely a feel-good band — their emotional range is a key part of their enduring appeal.

---

## 5  Combined Filter + GroupBy: High-Popularity Tracks by Era

Filter to tracks with Popularity ≥ 60 (moderately-to-highly popular on 2024 Spotify), then group by Era to see which period "ages" best in streaming.

In [ ]:
df_popular = df[df['Popularity'] >= 60].copy()

era_popularity = (
    df_popular.groupby('Era')['Popularity']
    .agg(['mean', 'count'])
    .round(2)
    .rename(columns={'mean': 'mean_popularity', 'count': 'n_songs'})
)

print(era_popularity)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(era_popularity.index, era_popularity['mean_popularity'],
            color=['royalblue', 'darkorange', 'green'])
axes[0].set_title('Mean Popularity (songs ≥60) by Era')
axes[0].set_ylabel('Mean Spotify Popularity')
axes[0].set_ylim(0, 100)

axes[1].bar(era_popularity.index, era_popularity['n_songs'],
            color=['royalblue', 'darkorange', 'green'])
axes[1].set_title('Count of High-Popularity Songs by Era')
axes[1].set_ylabel('Number of Songs')

for ax in axes:
    plt.sca(ax)
    plt.xticks(rotation=15)

plt.tight_layout()
plt.savefig('chart_era_popularity.png', dpi=100)
plt.show()

**Interpretation:** The late period (1969+) supplies the largest count of high-popularity streaming tracks, driven by *Abbey Road*'s towering canonical status. The psychedelic period (1966–68) features slightly higher *mean* popularity among its high-performing songs — consistent with the critical consensus that *Revolver* and *Sgt. Pepper's* are the band's artistic pinnacle. Early Beatles songs, though numerous, are somewhat less popular on contemporary streaming platforms, suggesting that casual listeners default to the later, more sonically polished output.

---

## 6  Summary

| Question | Method | Key Finding |
|----------|--------|-------------|
| Which songs are most critically canonical? | Filter + Sort | *Hey Jude*, *Let It Be*, *Come Together*, *Something* appear on 4+ independent lists |
| How are Spotify features distributed? | `pd.cut()` Bins + Bar | Most songs fall in the Medium-to-High Energy/Valence range; Instrumentalness skews very low |
| Do tempo markings cluster? | Custom Bin + Bar | The majority of songs fall in Moderato–Vivace (80–200 bpm) |
| Who writes the most popular songs? | GroupBy + Bar + Error bars | Lennon and McCartney are near-equal; Harrison competitive |
| What genres dominate? | Explode + GroupBy + Bar | Pop rock, Rock, Blues Rock are the top three |
| How do audio features vary by album? | GroupBy + Heatmap | A clear trajectory from upbeat/energetic early work to complex/experimental late period |
| Which era streams best today? | Filter + GroupBy + Bar | Late period has the most high-popularity tracks; psychedelic period has the highest mean |

### Methodological Reflection

- **Tidy data is non-negotiable.** Every `groupby` on a multi-value column (Genre, Moods) required a split-and-explode step first; skipping this would have produced meaningless groups.
- **`cut` vs `qcut`**: `cut` (equal-width) is best for domain-grounded categories (e.g. musical tempo markings). `qcut` (equal-count) is better for purely statistical comparisons where evenly-sized groups are needed.
- **Filtering before grouping** reduces noise and focuses the analysis — the era-popularity analysis would have been muddied if we had included every track regardless of streaming performance.
- **Normalisation in heatmaps** is essential when comparing features with different scales (Energy 0–1 vs Loudness –60–0 dB).

In [ ]:
print('All cells executed. Notebook complete.')